<a href="https://colab.research.google.com/github/buse-toklu/CENG467_Midterm/blob/main/question1_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q transformers datasets evaluate accelerate scikit-learn nltk

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import random

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
from sklearn.model_selection import train_test_split
import collections

# Load the SST-2 dataset from GLUE
dataset = load_dataset("glue", "sst2")

# Display dataset properties
print("--- SST-2 Dataset Properties ---")
print(dataset)
print("\nSample Data:")
print(f"Text: {dataset['train'][0]['sentence']}")
print(f"Label: {dataset['train'][0]['label']}")

# Use original validation set as TEST SET (only for final evaluation)
test_data = dataset['validation']
test_texts = test_data['sentence']
test_labels = test_data['label']

# Subsample from original train set
full_train = dataset['train'].shuffle(seed=42).select(range(5000))
all_texts = list(full_train['sentence']) # Convert to list
all_labels = list(full_train['label']) # Convert to list

# Split into train (%80) and validation (%20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    all_texts, all_labels, test_size=0.2, random_state=42
)

# Dataset stats
print("\n--- Dataset Split Sizes ---")
print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(f"Label distribution (train): {collections.Counter(train_labels)}")
print(f"Label distribution (val):   {collections.Counter(val_labels)}")
print(f"Label distribution (test):  {collections.Counter(test_labels)}")
print(f"Avg sentence length (train): {np.mean([len(t.split()) for t in train_texts]):.1f} words")
print(f"Avg sentence length (test):  {np.mean([len(t.split()) for t in test_texts]):.1f} words")

In [ ]:
print("=== TASK 2: Preprocessing & Tokenization ===")

stop_words = set(stopwords.words('english'))

nltk.download('punkt_tab')

# Strategy 1: Word-level (For Sparse / TF-IDF)
# Decisions: Lowercasing, stopword removal, punctuation removal.
def preprocess_sparse(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and t not in string.punctuation]
    return " ".join(tokens)

train_texts_sparse = [preprocess_sparse(t) for t in train_texts]
val_texts_sparse = [preprocess_sparse(t) for t in val_texts]

# Strategy 2: Subword-level (For Contextual / Neural)
# Decisions: Minimal normalization to preserve context, uses WordPiece algorithm, explicit truncation.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_text = "The movie was not entirely terrible, but lacking."
print(f"Original Text: {sample_text}")
print(f"1. Sparse Preprocessing (Word-level): {preprocess_sparse(sample_text)}")
print(f"2. Contextual Preprocessing (Subword-level): {tokenizer.tokenize(sample_text)}")

In [ ]:
print("=== TASK 3 & 4: Classical Model (TF-IDF + LR) ===")

# 1. Vectorization (Sparse Representation)
# We use max_features to limit the vocabulary and ngrams to capture word pairs
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(train_texts_sparse)
X_val_tfidf = vectorizer.transform(val_texts_sparse)

# 2. Model Training
# Logistic Regression serves as a strong baseline for high-dimensional sparse data
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, train_labels)

# 3. Train Performance
# Calculating metrics on training data to check for potential overfitting
train_preds = lr_model.predict(X_train_tfidf)
train_acc = accuracy_score(train_labels, train_preds)
train_f1 = f1_score(train_labels, train_preds, average='macro')

# 4. Validation Performance
# This represents the model's ability to generalize to unseen data
val_preds = lr_model.predict(X_val_tfidf)
val_acc = accuracy_score(val_labels, val_preds)
val_f1 = f1_score(val_labels, val_preds, average='macro')

# 5. Print Comparative Results
print("-" * 35)
print(f"TRAIN      -> Accuracy: {train_acc:.4f} | Macro-F1: {train_f1:.4f}")
print(f"VALIDATION -> Accuracy: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")
print("-" * 35)

# 6. Display Sample Predictions
# This allows for a quick qualitative check of model errors
print("\n[Sample Predictions]")
for i in range(3):
    status = "CORRECT" if val_preds[i] == val_labels[i] else "INCORRECT"
    print(f"Status: {status} | Text: {val_texts[i][:70]}...")
    print(f"    True Label: {val_labels[i]} | Predicted: {val_preds[i]}")
    print("-" * 20)

In [ ]:
print("=== TASK 2: Preprocessing Impact Experiment ===")

# Baseline: without stopword removal
def preprocess_no_stop(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in string.punctuation]
    return " ".join(tokens)

train_texts_no_stop = [preprocess_no_stop(t) for t in train_texts]
val_texts_no_stop   = [preprocess_no_stop(t) for t in val_texts]

vec_no_stop = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_no_stop = vec_no_stop.fit_transform(train_texts_no_stop)
X_val_no_stop   = vec_no_stop.transform(val_texts_no_stop)

lr_no_stop = LogisticRegression(max_iter=1000)
lr_no_stop.fit(X_train_no_stop, train_labels)
preds_no_stop = lr_no_stop.predict(X_val_no_stop)

acc_no_stop = accuracy_score(val_labels, preds_no_stop)
f1_no_stop  = f1_score(val_labels, preds_no_stop, average='macro')

# Define tfidf_acc and tfidf_f1 from previous cell's results (val_acc, val_f1)
tfidf_acc = val_acc
tfidf_f1 = val_f1

print(f"With stopword removal    -> Accuracy: {tfidf_acc:.4f} | Macro-F1: {tfidf_f1:.4f}")
print(f"Without stopword removal -> Accuracy: {acc_no_stop:.4f} | Macro-F1: {f1_no_stop:.4f}")

In [ ]:
print("=== TASK 3 & 4: Neural Model (BiLSTM) ===")

# --- Helper function for memory-safe evaluation ---
def get_all_predictions(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_X, batch_y in loader:
            outputs = model(batch_X.to(device))
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.numpy())
    return np.array(all_preds), np.array(all_labels)

# Create loaders for evaluation
train_eval_loader = DataLoader(LSTMDataset(X_train_seq, y_train_seq), batch_size=64)
val_eval_loader = DataLoader(LSTMDataset(X_val_seq, y_val_seq), batch_size=64)

# 1. Get Predictions
train_preds_lstm, train_labels_lstm = get_all_predictions(bilstm_model, train_eval_loader)
val_preds_lstm, val_labels_lstm = get_all_predictions(bilstm_model, val_eval_loader)

# 2. Calculate Metrics
train_acc_lstm = accuracy_score(train_labels_lstm, train_preds_lstm)
train_f1_lstm = f1_score(train_labels_lstm, train_preds_lstm, average='macro')

val_acc_lstm = accuracy_score(val_labels_lstm, val_preds_lstm)
val_f1_lstm = f1_score(val_labels_lstm, val_preds_lstm, average='macro')

# 3. Print Comparative Results
print("-" * 35)
print(f"TRAIN      -> Accuracy: {train_acc_lstm:.4f} | Macro-F1: {train_f1_lstm:.4f}")
print(f"VALIDATION -> Accuracy: {val_acc_lstm:.4f} | Macro-F1: {val_f1_lstm:.4f}")
print("-" * 35)

# 4. Display Sample Predictions (English Status)
print("\n[Sample Predictions]")
for i in range(3):
    status = "CORRECT" if val_preds_lstm[i] == val_labels_lstm[i] else "INCORRECT"
    print(f"Status: {status} | Text: {val_texts[i][:70]}...")
    print(f"    True Label: {val_labels_lstm[i]} | Predicted: {val_preds_lstm[i]}")
    print("-" * 20)

In [ ]:
print("=== TASK 3 & 4: Contextual Model (DistilBERT) ===")

from datasets import Dataset

# Convert to HuggingFace Dataset format
train_dataset_hf = Dataset.from_dict({"sentence": list(train_texts), "label": list(train_labels)})
val_dataset_hf   = Dataset.from_dict({"sentence": list(val_texts),   "label": list(val_labels)})

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_dataset_hf.map(tokenize_function, batched=True)
tokenized_val   = val_dataset_hf.map(tokenize_function, batched=True)

# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

distilbert_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

# --- 1. Evaluation on Training Set (For Generalization Analysis) ---
print("Evaluating on Training Set...")
train_results = trainer.evaluate(tokenized_train)

# --- 2. Evaluation on Validation Set ---
print("Evaluating on Validation Set...")
val_results = trainer.evaluate(tokenized_val)

# --- 3. Print Comparative Results ---
print("-" * 35)
print(f"TRAIN      -> Accuracy: {train_results['eval_accuracy']:.4f} | Macro-F1: {train_results['eval_f1_macro']:.4f}")
print(f"VALIDATION -> Accuracy: {val_results['eval_accuracy']:.4f} | Macro-F1: {val_results['eval_f1_macro']:.4f}")
print("-" * 35)

# --- 4. Sample Predictions ---
print("\n[Sample Predictions]")
raw_pred, _, _ = trainer.predict(tokenized_val)
distilbert_preds = np.argmax(raw_pred, axis=1)

val_labels_list = list(val_labels)
val_texts_list  = list(val_texts)

for i in range(3):
    status = "CORRECT" if distilbert_preds[i] == val_labels_list[i] else "INCORRECT"
    print(f"Status: {status} | Text: {val_texts_list[i][:70]}...")
    print(f"    True Label: {val_labels_list[i]} | Predicted: {distilbert_preds[i]}")
    print("-" * 20)

In [ ]:
print("=== TASK 4: Error Analysis (All Models) ===")

label_map = {0: "Negative", 1: "Positive"}

# --- TF-IDF + LR ---
lr_df = pd.DataFrame({
    'text': list(val_texts),
    'true_label': list(val_labels),
    'predicted_label': val_preds
})
lr_misclassified = lr_df[lr_df['true_label'] != lr_df['predicted_label']]
print(f"TF-IDF+LR  misclassified: {len(lr_misclassified)} / {len(lr_df)}")

# --- BiLSTM (recompute on current val set) ---
X_val_seq_new = encode_for_lstm(list(val_texts))
bilstm_model.eval()
with torch.no_grad():
    bilstm_preds_new = torch.argmax(bilstm_model(X_val_seq_new.to(device)), dim=1).cpu().numpy()

bilstm_df = pd.DataFrame({
    'text': list(val_texts),
    'true_label': list(val_labels),
    'predicted_label': bilstm_preds_new
})
bilstm_misclassified = bilstm_df[bilstm_df['true_label'] != bilstm_df['predicted_label']]
print(f"BiLSTM     misclassified: {len(bilstm_misclassified)} / {len(bilstm_df)}")

# --- DistilBERT ---
predictions, labels, _ = trainer.predict(tokenized_val)
predicted_classes = np.argmax(predictions, axis=1)
bert_df = pd.DataFrame({
    'text': list(val_texts),
    'true_label': labels,
    'predicted_label': predicted_classes
})
bert_misclassified = bert_df[bert_df['true_label'] != bert_df['predicted_label']]
print(f"DistilBERT misclassified: {len(bert_misclassified)} / {len(bert_df)}")

# --- 5 örnek sadece DistilBERT'ten (en iyi model) ---
print("\n--- 5 Misclassified Examples (DistilBERT) ---")
sample_errors = bert_misclassified.sample(n=5, random_state=42).copy()
sample_errors['true_label'] = sample_errors['true_label'].map(label_map)
sample_errors['predicted_label'] = sample_errors['predicted_label'].map(label_map)

pd.set_option('display.max_colwidth', None)
display(sample_errors)

print("\n--- Guidelines for Your Report ---")
print("Look at the 5 sentences above. Identify patterns such as:")
print("1. Complex Negation (e.g., 'not exactly a masterpiece')")
print("2. Sarcasm / Irony")
print("3. Mixed Sentiment (Starts positive, ends negative)")
print("4. Rare Idioms or Domain-Specific Jargon")

In [ ]:
print("\n" + "="*55)
print("         FINAL PERFORMANCE COMPARISON")
print("="*55)
print(f"{'Model':<30} {'Accuracy':>10} {'Macro-F1':>10}")
print("-"*55)
print(f"{'1. Sparse  (TF-IDF + LR)':<30} {val_acc:>10.4f} {val_f1:>10.4f}")
print(f"{'2. Dense   (BiLSTM)':<30} {bilstm_acc:>10.4f} {bilstm_f1:>10.4f}")
print(f"{'3. Contextual (DistilBERT)':<30} {val_results['eval_accuracy']:>10.4f} {val_results['eval_f1_macro']:>10.4f}")
print("="*55)

In [ ]:
print("=== FINAL EVALUATION ON TEST SET ===")
print("(Test set used only once for final evaluation)\n")

# --- TF-IDF + LR ---
test_texts_sparse = [preprocess_sparse(t) for t in test_texts]
X_test_tfidf = vectorizer.transform(test_texts_sparse)
test_preds_lr = lr_model.predict(X_test_tfidf)
test_acc_lr = accuracy_score(test_labels, test_preds_lr)
test_f1_lr  = f1_score(test_labels, test_preds_lr, average='macro')

# --- BiLSTM ---
X_test_seq = encode_for_lstm(list(test_texts))
bilstm_model.eval()
with torch.no_grad():
    test_preds_bilstm = torch.argmax(bilstm_model(X_test_seq.to(device)), dim=1).cpu().numpy()
test_acc_bilstm = accuracy_score(test_labels, test_preds_bilstm)
test_f1_bilstm  = f1_score(test_labels, test_preds_bilstm, average='macro')

# --- DistilBERT ---
from datasets import Dataset as HFDataset
test_dataset_hf = HFDataset.from_dict({"sentence": list(test_texts), "label": list(test_labels)})
tokenized_test  = test_dataset_hf.map(tokenize_function, batched=True)
bert_test_results = trainer.evaluate(tokenized_test)
test_acc_bert = bert_test_results['eval_accuracy']
test_f1_bert  = bert_test_results['eval_f1_macro']

# --- Print Results ---
print("="*55)
print("         TEST SET PERFORMANCE (FINAL)")
print("="*55)
print(f"{'Model':<30} {'Accuracy':>10} {'Macro-F1':>10}")
print("-"*55)
print(f"{'1. Sparse  (TF-IDF + LR)':<30} {test_acc_lr:>10.4f} {test_f1_lr:>10.4f}")
print(f"{'2. Dense   (BiLSTM)':<30} {test_acc_bilstm:>10.4f} {test_f1_bilstm:>10.4f}")
print(f"{'3. Contextual (DistilBERT)':<30} {test_acc_bert:>10.4f} {test_f1_bert:>10.4f}")
print("="*55)